# 🚀 基础推理和批处理

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/nano-vllm-learning/blob/main/notebooks/01_基础推理和批处理.ipynb)

## 📋 学习目标

本Notebook将带你深入了解LLM推理的核心概念和批处理优化技术：

### 🎯 核心内容
1. **Tokenization深度解析** - 理解不同分词策略的影响
2. **推理过程详解** - 从输入到输出的完整流程
3. **批处理优化** - 静态、动态和连续批处理策略
4. **性能分析** - 吞吐量、延迟和内存使用优化
5. **高级批处理技术** - 优先级队列和智能调度

### 🔧 技术要点
- 词汇表构建和token映射
- 前向传播和采样策略
- 内存管理和计算优化
- 请求调度和负载均衡

让我们开始这个激动人心的学习之旅！🎉

## 🔧 环境设置

首先设置我们的实验环境，包括必要的依赖和工具。

In [ ]:
# 检测是否在Colab环境中运行
try:
    import google.colab
    IN_COLAB = True
    print("🔍 检测到Google Colab环境")
except ImportError:
    IN_COLAB = False
    print("🔍 检测到本地环境")

# 如果在Colab中，克隆项目仓库
if IN_COLAB:
    print("📥 正在克隆nano-vLLM项目...")
    !git clone https://github.com/your-username/nano-vllm-learning.git
    %cd nano-vllm-learning
    print("✅ 项目克隆完成")

# 安装必要的依赖
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# 基础依赖
required_packages = [
    "torch",
    "numpy",
    "matplotlib",
    "seaborn",
    "tqdm",
    "transformers"
]

print("📦 安装必要的依赖包...")
for package in required_packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✅ {package} 已安装")
    except ImportError:
        print(f"📥 正在安装 {package}...")
        install_package(package)
        print(f"✅ {package} 安装完成")

print("\n🎉 环境设置完成！")

In [ ]:
# 导入必要的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional, Tuple, Any, Union
from dataclasses import dataclass, field
from enum import Enum
import time
import threading
import queue
import json
import heapq
from collections import defaultdict, deque
import warnings
warnings.filterwarnings('ignore')

# 设置绘图样式
plt.style.use('default')
sns.set_palette("husl")

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

print("📚 库导入完成")
print(f"🔥 PyTorch版本: {torch.__version__}")
print(f"🎯 CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU设备: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 🔤 深入理解Tokenization

Tokenization是LLM推理的第一步，也是最关键的步骤之一。让我们深入探索不同的分词策略及其对性能的影响。

In [ ]:
class AdvancedTokenizer:
    """高级分词器 - 支持多种分词策略"""
    
    def __init__(self, vocab_size: int = 10000):
        self.vocab_size = vocab_size
        self.word_to_id = {}
        self.id_to_word = {}
        self.char_to_id = {}
        self.id_to_char = {}
        self.subword_to_id = {}
        self.id_to_subword = {}
        
        # 特殊token
        self.special_tokens = {
            '<pad>': 0,
            '<unk>': 1,
            '<bos>': 2,
            '<eos>': 3
        }
        
        self._build_vocabularies()
    
    def _build_vocabularies(self):
        """构建不同类型的词汇表"""
        # 字符级词汇表
        chars = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?;:'-\"()[]{}")
        for i, char in enumerate(chars):
            self.char_to_id[char] = i + len(self.special_tokens)
            self.id_to_char[i + len(self.special_tokens)] = char
        
        # 词级词汇表（简化版）
        common_words = [
            "the", "and", "is", "in", "to", "of", "a", "that", "it", "with",
            "for", "as", "was", "on", "are", "you", "this", "be", "at", "have",
            "from", "or", "one", "had", "by", "word", "but", "not", "what", "all",
            "were", "they", "we", "when", "your", "can", "said", "there", "each", "which",
            "she", "do", "how", "their", "if", "will", "up", "other", "about", "out"
        ]
        
        for i, word in enumerate(common_words):
            self.word_to_id[word] = i + len(self.special_tokens)
            self.id_to_word[i + len(self.special_tokens)] = word
        
        # 子词级词汇表（BPE风格）
        subwords = [
            "##ing", "##ed", "##er", "##ly", "##tion", "##al", "##ent", "##ant",
            "un##", "re##", "pre##", "dis##", "##able", "##ful", "##less", "##ness"
        ]
        
        for i, subword in enumerate(subwords):
            self.subword_to_id[subword] = i + len(self.special_tokens)
            self.id_to_subword[i + len(self.special_tokens)] = subword
    
    def tokenize_word_level(self, text: str) -> List[int]:
        """词级分词"""
        words = text.lower().split()
        tokens = [self.special_tokens['<bos>']]
        
        for word in words:
            # 移除标点符号
            clean_word = ''.join(c for c in word if c.isalnum())
            if clean_word in self.word_to_id:
                tokens.append(self.word_to_id[clean_word])
            else:
                tokens.append(self.special_tokens['<unk>'])
        
        tokens.append(self.special_tokens['<eos>'])
        return tokens
    
    def tokenize_char_level(self, text: str) -> List[int]:
        """字符级分词"""
        tokens = [self.special_tokens['<bos>']]
        
        for char in text:
            if char in self.char_to_id:
                tokens.append(self.char_to_id[char])
            else:
                tokens.append(self.special_tokens['<unk>'])
        
        tokens.append(self.special_tokens['<eos>'])
        return tokens
    
    def compare_tokenization_strategies(self, text: str) -> Dict[str, Any]:
        """比较不同分词策略"""
        strategies = {
            'word_level': self.tokenize_word_level,
            'char_level': self.tokenize_char_level
        }
        
        results = {}
        
        for name, tokenize_func in strategies.items():
            start_time = time.time()
            tokens = tokenize_func(text)
            tokenization_time = time.time() - start_time
            
            results[name] = {
                'tokens': tokens,
                'num_tokens': len(tokens),
                'tokenization_time': tokenization_time,
                'compression_ratio': len(text) / len(tokens)
            }
        
        return results

# 创建高级分词器
tokenizer = AdvancedTokenizer()

# 测试文本
test_text = "The advanced tokenization strategies are working efficiently and processing text data."

# 比较不同分词策略
tokenization_results = tokenizer.compare_tokenization_strategies(test_text)

print("🔤 分词策略比较结果:")
print("=" * 60)
print(f"原始文本: '{test_text}'")
print(f"文本长度: {len(test_text)} 字符\n")

for strategy, result in tokenization_results.items():
    print(f"📊 {strategy.replace('_', ' ').title()}:")
    print(f"   Token数量: {result['num_tokens']}")
    print(f"   分词时间: {result['tokenization_time']:.6f}s")
    print(f"   压缩比: {result['compression_ratio']:.2f}")
    print(f"   前10个tokens: {result['tokens'][:10]}")
    print()

## 🧠 推理引擎实现

现在让我们实现一个简化的推理引擎，理解从token到输出的完整流程。

In [ ]:
@dataclass
class InferenceRequest:
    """推理请求数据结构"""
    request_id: str
    prompt: str
    max_tokens: int = 100
    temperature: float = 1.0
    top_p: float = 0.9
    priority: int = 1
    timestamp: float = field(default_factory=time.time)

@dataclass
class InferenceResult:
    """推理结果数据结构"""
    request_id: str
    generated_text: str
    tokens_generated: int
    inference_time: float
    tokens_per_second: float

class SimpleInferenceEngine:
    """简化的推理引擎"""
    
    def __init__(self, vocab_size: int = 10000, hidden_size: int = 512):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.tokenizer = AdvancedTokenizer(vocab_size)
        
        # 简化的模型参数（随机初始化）
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.transformer_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=8,
            dim_feedforward=2048,
            batch_first=True
        )
        self.output_projection = nn.Linear(hidden_size, vocab_size)
        
        # 设置为评估模式
        self.embedding.eval()
        self.transformer_layer.eval()
        self.output_projection.eval()
    
    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """前向传播"""
        # 嵌入层
        embeddings = self.embedding(input_ids)
        
        # Transformer层
        hidden_states = self.transformer_layer(embeddings)
        
        # 输出投影
        logits = self.output_projection(hidden_states)
        
        return logits
    
    def sample_next_token(self, logits: torch.Tensor, temperature: float = 1.0, top_p: float = 0.9) -> int:
        """采样下一个token"""
        # 应用温度
        logits = logits / temperature
        
        # 应用top-p采样
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        
        # 计算累积概率
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        
        # 找到top-p截断点
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
        
        # 将被移除的token概率设为0
        indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
        probs[indices_to_remove] = 0
        
        # 重新归一化
        probs = probs / probs.sum(dim=-1, keepdim=True)
        
        # 采样
        next_token = torch.multinomial(probs, num_samples=1)
        return next_token.item()
    
    def generate_text(self, request: InferenceRequest) -> InferenceResult:
        """生成文本"""
        start_time = time.time()
        
        # 分词
        input_tokens = self.tokenizer.tokenize_word_level(request.prompt)
        input_ids = torch.tensor([input_tokens], dtype=torch.long)
        
        generated_tokens = []
        
        with torch.no_grad():
            for _ in range(request.max_tokens):
                # 前向传播
                logits = self.forward(input_ids)
                
                # 获取最后一个位置的logits
                next_token_logits = logits[0, -1, :]
                
                # 采样下一个token
                next_token = self.sample_next_token(
                    next_token_logits,
                    temperature=request.temperature,
                    top_p=request.top_p
                )
                
                # 检查是否为结束token
                if next_token == self.tokenizer.special_tokens['<eos>']:
                    break
                
                generated_tokens.append(next_token)
                
                # 更新输入序列
                new_token = torch.tensor([[next_token]], dtype=torch.long)
                input_ids = torch.cat([input_ids, new_token], dim=1)
        
        # 计算推理时间和性能指标
        inference_time = time.time() - start_time
        tokens_generated = len(generated_tokens)
        tokens_per_second = tokens_generated / inference_time if inference_time > 0 else 0
        
        # 简化的文本解码（这里只是示例）
        generated_text = f"Generated {tokens_generated} tokens"
        
        return InferenceResult(
            request_id=request.request_id,
            generated_text=generated_text,
            tokens_generated=tokens_generated,
            inference_time=inference_time,
            tokens_per_second=tokens_per_second
        )

# 创建推理引擎
inference_engine = SimpleInferenceEngine()

# 创建测试请求
test_request = InferenceRequest(
    request_id="test_001",
    prompt="The future of artificial intelligence",
    max_tokens=50,
    temperature=0.8,
    top_p=0.9
)

# 执行推理
result = inference_engine.generate_text(test_request)

print("🧠 推理引擎测试结果:")
print("=" * 50)
print(f"请求ID: {result.request_id}")
print(f"生成文本: {result.generated_text}")
print(f"生成token数: {result.tokens_generated}")
print(f"推理时间: {result.inference_time:.4f}s")
print(f"吞吐量: {result.tokens_per_second:.2f} tokens/s")

## 📦 批处理优化策略

批处理是提高LLM推理效率的关键技术。让我们实现和比较不同的批处理策略。

In [ ]:
class BatchingStrategy(Enum):
    """批处理策略枚举"""
    STATIC = "static"
    DYNAMIC = "dynamic"
    CONTINUOUS = "continuous"

class BatchProcessor:
    """批处理器"""
    
    def __init__(self, inference_engine: SimpleInferenceEngine, max_batch_size: int = 8):
        self.inference_engine = inference_engine
        self.max_batch_size = max_batch_size
        self.request_queue = queue.Queue()
        self.active_requests = {}
        
    def add_request(self, request: InferenceRequest):
        """添加推理请求"""
        self.request_queue.put(request)
    
    def static_batching(self, requests: List[InferenceRequest]) -> List[InferenceResult]:
        """静态批处理 - 等待批次填满后一起处理"""
        results = []
        
        # 按批次大小分组处理
        for i in range(0, len(requests), self.max_batch_size):
            batch = requests[i:i + self.max_batch_size]
            batch_start_time = time.time()
            
            # 并行处理批次中的所有请求（简化实现）
            batch_results = []
            for request in batch:
                result = self.inference_engine.generate_text(request)
                batch_results.append(result)
            
            batch_time = time.time() - batch_start_time
            
            # 更新批处理时间信息
            for result in batch_results:
                result.inference_time = batch_time / len(batch)  # 平均时间
                result.tokens_per_second = result.tokens_generated / result.inference_time
            
            results.extend(batch_results)
        
        return results
    
    def dynamic_batching(self, requests: List[InferenceRequest], timeout: float = 0.1) -> List[InferenceResult]:
        """动态批处理 - 在超时时间内收集请求"""
        results = []
        remaining_requests = requests.copy()
        
        while remaining_requests:
            batch = []
            batch_start_time = time.time()
            
            # 在超时时间内收集请求
            while len(batch) < self.max_batch_size and remaining_requests:
                if time.time() - batch_start_time > timeout and batch:
                    break
                
                batch.append(remaining_requests.pop(0))
                
                # 模拟等待新请求的时间
                if len(batch) < self.max_batch_size and remaining_requests:
                    time.sleep(0.01)  # 10ms等待
            
            # 处理当前批次
            if batch:
                batch_results = []
                for request in batch:
                    result = self.inference_engine.generate_text(request)
                    batch_results.append(result)
                
                results.extend(batch_results)
        
        return results
    
    def continuous_batching(self, requests: List[InferenceRequest]) -> List[InferenceResult]:
        """连续批处理 - 请求完成后立即添加新请求"""
        results = []
        request_iter = iter(requests)
        active_batch = []
        
        # 初始化第一个批次
        try:
            for _ in range(self.max_batch_size):
                active_batch.append(next(request_iter))
        except StopIteration:
            pass
        
        while active_batch:
            # 处理当前批次
            batch_results = []
            completed_indices = []
            
            for i, request in enumerate(active_batch):
                result = self.inference_engine.generate_text(request)
                batch_results.append(result)
                completed_indices.append(i)
            
            results.extend(batch_results)
            
            # 移除已完成的请求并添加新请求
            for i in reversed(completed_indices):
                active_batch.pop(i)
                try:
                    active_batch.append(next(request_iter))
                except StopIteration:
                    pass
        
        return results
    
    def compare_batching_strategies(self, requests: List[InferenceRequest]) -> Dict[str, Any]:
        """比较不同批处理策略的性能"""
        strategies = {
            'static': self.static_batching,
            'dynamic': self.dynamic_batching,
            'continuous': self.continuous_batching
        }
        
        comparison_results = {}
        
        for strategy_name, strategy_func in strategies.items():
            start_time = time.time()
            
            # 复制请求列表以避免修改原始数据
            requests_copy = [InferenceRequest(
                request_id=f"{req.request_id}_{strategy_name}",
                prompt=req.prompt,
                max_tokens=req.max_tokens,
                temperature=req.temperature,
                top_p=req.top_p
            ) for req in requests]
            
            results = strategy_func(requests_copy)
            total_time = time.time() - start_time
            
            # 计算性能指标
            total_tokens = sum(r.tokens_generated for r in results)
            avg_latency = sum(r.inference_time for r in results) / len(results)
            throughput = total_tokens / total_time
            
            comparison_results[strategy_name] = {
                'total_time': total_time,
                'total_tokens': total_tokens,
                'avg_latency': avg_latency,
                'throughput': throughput,
                'num_requests': len(results)
            }
        
        return comparison_results

# 创建批处理器
batch_processor = BatchProcessor(inference_engine, max_batch_size=4)

# 创建测试请求列表
test_requests = [
    InferenceRequest(f"req_{i}", f"Prompt {i}: The future of AI", max_tokens=30)
    for i in range(12)
]

# 比较批处理策略
batching_comparison = batch_processor.compare_batching_strategies(test_requests)

print("📦 批处理策略性能比较:")
print("=" * 70)

for strategy, metrics in batching_comparison.items():
    print(f"\n🔄 {strategy.title()} Batching:")
    print(f"   总处理时间: {metrics['total_time']:.4f}s")
    print(f"   总生成tokens: {metrics['total_tokens']}")
    print(f"   平均延迟: {metrics['avg_latency']:.4f}s")
    print(f"   吞吐量: {metrics['throughput']:.2f} tokens/s")
    print(f"   处理请求数: {metrics['num_requests']}")

## 📊 性能分析和可视化

让我们分析和可视化不同策略的性能表现。

In [ ]:
def create_performance_visualization(batching_results: Dict[str, Any], tokenization_results: Dict[str, Any]):
    """创建性能分析可视化"""
    
    # 创建子图
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('🚀 LLM推理性能分析', fontsize=16, fontweight='bold')
    
    # 1. 批处理策略吞吐量比较
    strategies = list(batching_results.keys())
    throughputs = [batching_results[s]['throughput'] for s in strategies]
    
    bars1 = ax1.bar(strategies, throughputs, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    ax1.set_title('📦 批处理策略吞吐量比较', fontweight='bold')
    ax1.set_ylabel('吞吐量 (tokens/s)')
    ax1.set_xlabel('批处理策略')
    
    # 添加数值标签
    for bar, value in zip(bars1, throughputs):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{value:.1f}', ha='center', va='bottom', fontweight='bold')
    
    # 2. 批处理策略延迟比较
    latencies = [batching_results[s]['avg_latency'] for s in strategies]
    
    bars2 = ax2.bar(strategies, latencies, color=['#FF9F43', '#6C5CE7', '#A29BFE'])
    ax2.set_title('⏱️ 批处理策略平均延迟比较', fontweight='bold')
    ax2.set_ylabel('平均延迟 (s)')
    ax2.set_xlabel('批处理策略')
    
    # 添加数值标签
    for bar, value in zip(bars2, latencies):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # 3. 分词策略比较
    tokenization_strategies = list(tokenization_results.keys())
    token_counts = [tokenization_results[s]['num_tokens'] for s in tokenization_strategies]
    compression_ratios = [tokenization_results[s]['compression_ratio'] for s in tokenization_strategies]
    
    x_pos = np.arange(len(tokenization_strategies))
    width = 0.35
    
    bars3_1 = ax3.bar(x_pos - width/2, token_counts, width, label='Token数量', color='#74B9FF')
    ax3_twin = ax3.twinx()
    bars3_2 = ax3_twin.bar(x_pos + width/2, compression_ratios, width, label='压缩比', color='#FD79A8')
    
    ax3.set_title('🔤 分词策略效率比较', fontweight='bold')
    ax3.set_xlabel('分词策略')
    ax3.set_ylabel('Token数量', color='#74B9FF')
    ax3_twin.set_ylabel('压缩比', color='#FD79A8')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels([s.replace('_', ' ').title() for s in tokenization_strategies])
    
    # 添加图例
    lines1, labels1 = ax3.get_legend_handles_labels()
    lines2, labels2 = ax3_twin.get_legend_handles_labels()
    ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    # 4. 性能效率雷达图
    categories = ['吞吐量', '低延迟', '资源利用率', '可扩展性']
    
    # 归一化性能数据（0-1范围）
    max_throughput = max(throughputs)
    min_latency = min(latencies)
    
    performance_data = {}
    for i, strategy in enumerate(strategies):
        performance_data[strategy] = [
            throughputs[i] / max_throughput,  # 吞吐量（越高越好）
            min_latency / latencies[i],       # 低延迟（越低越好，所以取倒数）
            0.7 + i * 0.1,                   # 资源利用率（模拟数据）
            0.6 + i * 0.15                   # 可扩展性（模拟数据）
        ]
    
    # 计算角度
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]  # 闭合图形
    
    ax4 = plt.subplot(2, 2, 4, projection='polar')
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    for i, (strategy, values) in enumerate(performance_data.items()):
        values += values[:1]  # 闭合图形
        ax4.plot(angles, values, 'o-', linewidth=2, label=strategy.title(), color=colors[i])
        ax4.fill(angles, values, alpha=0.25, color=colors[i])
    
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(categories)
    ax4.set_ylim(0, 1)
    ax4.set_title('🎯 综合性能雷达图', fontweight='bold', pad=20)
    ax4.legend(loc='upper right', bbox_to_anchor=(1.2, 1.0))
    
    plt.tight_layout()
    plt.show()
    
    # 打印性能总结
    print("\n📈 性能分析总结:")
    print("=" * 50)
    
    best_throughput = max(batching_results.items(), key=lambda x: x[1]['throughput'])
    best_latency = min(batching_results.items(), key=lambda x: x[1]['avg_latency'])
    
    print(f"🏆 最佳吞吐量: {best_throughput[0].title()} ({best_throughput[1]['throughput']:.2f} tokens/s)")
    print(f"⚡ 最低延迟: {best_latency[0].title()} ({best_latency[1]['avg_latency']:.4f}s)")
    
    best_compression = max(tokenization_results.items(), key=lambda x: x[1]['compression_ratio'])
    print(f"🔤 最佳压缩比: {best_compression[0].replace('_', ' ').title()} ({best_compression[1]['compression_ratio']:.2f})")

# 创建性能可视化
create_performance_visualization(batching_comparison, tokenization_results)

## 🔧 高级优化技术

让我们探索一些高级的优化技术，包括优先级调度和智能批处理。

In [ ]:
class PriorityScheduler:
    """优先级调度器"""
    
    def __init__(self, inference_engine: SimpleInferenceEngine):
        self.inference_engine = inference_engine
        self.priority_queue = []
        self.request_counter = 0
        
    def add_request(self, request: InferenceRequest):
        """添加请求到优先级队列"""
        # 使用负优先级实现最大堆（优先级越高越先处理）
        priority_score = self._calculate_priority_score(request)
        heapq.heappush(self.priority_queue, (-priority_score, self.request_counter, request))
        self.request_counter += 1
    
    def _calculate_priority_score(self, request: InferenceRequest) -> float:
        """计算请求的优先级分数"""
        base_priority = request.priority
        
        # 考虑等待时间（越久等待优先级越高）
        wait_time = time.time() - request.timestamp
        wait_bonus = min(wait_time * 0.1, 2.0)  # 最多加2分
        
        # 考虑请求长度（短请求优先级稍高）
        length_penalty = len(request.prompt) / 1000.0
        
        # 考虑生成长度（短生成优先级稍高）
        generation_penalty = request.max_tokens / 1000.0
        
        total_score = base_priority + wait_bonus - length_penalty - generation_penalty
        return max(total_score, 0.1)  # 确保最小优先级
    
    def get_next_batch(self, batch_size: int) -> List[InferenceRequest]:
        """获取下一个批次的请求"""
        batch = []
        
        while len(batch) < batch_size and self.priority_queue:
            _, _, request = heapq.heappop(self.priority_queue)
            batch.append(request)
        
        return batch
    
    def process_with_priority(self, requests: List[InferenceRequest], batch_size: int = 4) -> List[InferenceResult]:
        """使用优先级调度处理请求"""
        # 添加所有请求到优先级队列
        for request in requests:
            self.add_request(request)
        
        results = []
        
        while self.priority_queue:
            # 获取下一个批次
            batch = self.get_next_batch(batch_size)
            
            if not batch:
                break
            
            # 处理批次
            batch_start_time = time.time()
            batch_results = []
            
            for request in batch:
                result = self.inference_engine.generate_text(request)
                batch_results.append(result)
            
            batch_time = time.time() - batch_start_time
            
            # 更新批处理时间
            for result in batch_results:
                result.inference_time = batch_time / len(batch)
                result.tokens_per_second = result.tokens_generated / result.inference_time
            
            results.extend(batch_results)
        
        return results

class AdaptiveBatchProcessor:
    """自适应批处理器"""
    
    def __init__(self, inference_engine: SimpleInferenceEngine, initial_batch_size: int = 4):
        self.inference_engine = inference_engine
        self.current_batch_size = initial_batch_size
        self.min_batch_size = 1
        self.max_batch_size = 16
        self.performance_history = deque(maxlen=10)
        
    def _adjust_batch_size(self, current_throughput: float):
        """根据性能调整批次大小"""
        if len(self.performance_history) < 2:
            self.performance_history.append(current_throughput)
            return
        
        # 计算性能趋势
        recent_avg = sum(list(self.performance_history)[-3:]) / min(3, len(self.performance_history))
        
        if current_throughput > recent_avg * 1.1:  # 性能提升
            if self.current_batch_size < self.max_batch_size:
                self.current_batch_size = min(self.current_batch_size + 1, self.max_batch_size)
        elif current_throughput < recent_avg * 0.9:  # 性能下降
            if self.current_batch_size > self.min_batch_size:
                self.current_batch_size = max(self.current_batch_size - 1, self.min_batch_size)
        
        self.performance_history.append(current_throughput)
    
    def adaptive_process(self, requests: List[InferenceRequest]) -> List[InferenceResult]:
        """自适应批处理"""
        results = []
        remaining_requests = requests.copy()
        
        while remaining_requests:
            # 获取当前批次
            batch_size = min(self.current_batch_size, len(remaining_requests))
            batch = remaining_requests[:batch_size]
            remaining_requests = remaining_requests[batch_size:]
            
            # 处理批次
            batch_start_time = time.time()
            batch_results = []
            
            for request in batch:
                result = self.inference_engine.generate_text(request)
                batch_results.append(result)
            
            batch_time = time.time() - batch_start_time
            
            # 计算当前吞吐量
            total_tokens = sum(r.tokens_generated for r in batch_results)
            current_throughput = total_tokens / batch_time
            
            # 调整批次大小
            self._adjust_batch_size(current_throughput)
            
            # 更新结果
            for result in batch_results:
                result.inference_time = batch_time / len(batch)
                result.tokens_per_second = result.tokens_generated / result.inference_time
            
            results.extend(batch_results)
        
        return results

# 创建高级调度器和处理器
priority_scheduler = PriorityScheduler(inference_engine)
adaptive_processor = AdaptiveBatchProcessor(inference_engine)

# 创建不同优先级的测试请求
high_priority_requests = [
    InferenceRequest(f"high_{i}", "Urgent: Quick response needed", max_tokens=20, priority=5)
    for i in range(3)
]

medium_priority_requests = [
    InferenceRequest(f"med_{i}", "Standard request for processing", max_tokens=40, priority=3)
    for i in range(6)
]

low_priority_requests = [
    InferenceRequest(f"low_{i}", "Background task with lower priority", max_tokens=60, priority=1)
    for i in range(3)
]

all_requests = high_priority_requests + medium_priority_requests + low_priority_requests
np.random.shuffle(all_requests)  # 随机打乱顺序

# 测试优先级调度
print("🎯 优先级调度测试:")
print("=" * 40)

priority_start_time = time.time()
priority_results = priority_scheduler.process_with_priority(all_requests.copy())
priority_total_time = time.time() - priority_start_time

print(f"处理顺序 (按优先级):")
for i, result in enumerate(priority_results[:6]):  # 显示前6个
    request_id = result.request_id
    priority_type = request_id.split('_')[0]
    print(f"  {i+1}. {request_id} ({priority_type} priority)")

print(f"\n总处理时间: {priority_total_time:.4f}s")
print(f"平均吞吐量: {sum(r.tokens_generated for r in priority_results) / priority_total_time:.2f} tokens/s")

# 测试自适应批处理
print("\n🔄 自适应批处理测试:")
print("=" * 40)

adaptive_start_time = time.time()
adaptive_results = adaptive_processor.adaptive_process(all_requests.copy())
adaptive_total_time = time.time() - adaptive_start_time

print(f"最终批次大小: {adaptive_processor.current_batch_size}")
print(f"性能历史: {list(adaptive_processor.performance_history)[-3:]}")
print(f"总处理时间: {adaptive_total_time:.4f}s")
print(f"平均吞吐量: {sum(r.tokens_generated for r in adaptive_results) / adaptive_total_time:.2f} tokens/s")

## 🎓 恭喜完成！

您已经成功完成了基础推理和批处理的学习！现在您应该能够：

✅ **理解 Tokenization**：掌握不同分词策略的特点和适用场景
✅ **掌握推理流程**：了解从输入到输出的完整推理过程
✅ **优化批处理**：实现和比较不同的批处理策略
✅ **性能分析**：评估和优化推理性能指标
✅ **高级优化**：实现优先级调度和自适应批处理

### 🔗 相关Notebooks
- **下一步**: [02_高级调度和内存管理.ipynb](./02_高级调度和内存管理.ipynb)
- **基础**: [00_环境设置和快速开始.ipynb](./00_环境设置和快速开始.ipynb)
- **进阶**: [03_完整推理流程和端到端系统.ipynb](./03_完整推理流程和端到端系统.ipynb)

### 💡 实践建议

1. **修改参数**：尝试不同的批次大小、温度参数等
2. **添加新策略**：实现您自己的批处理优化算法
3. **性能测试**：在不同的硬件配置下测试性能
4. **可视化分析**：创建更多的性能分析图表

继续您的 nano-vLLM 学习之旅！🚀